# exp047 AnuraSet Extract (CPU + Internet)

**Purpose**: AnuraSet v2 から BC2026 Amphibia species filter extract

**Input**:
- `bengtlueers/anuraset-v2-raw` (raw audio only)
- Internet ON: GitHub から annotation CSV DL

**Output**: `maekeso/birdclef2026-exp047-anuraset-extracted`

**Structure**:
- Audio: `AnuraSet_v2.0.0_raw/{site}/{site}_{date}_{time}.wav` (multi-species soundscape)
- Annotation: GitHub <https://github.com/soundclim/anuraset> から DL

**Match**: annotation CSV の scientific_name → BC2026 Amphibia


In [ ]:
import os, sys, json, time, re, shutil
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import urllib.request

OUT_DIR = Path("/kaggle/working/extracted")
OUT_DIR.mkdir(exist_ok=True, parents=True)
AUDIO_DIR = OUT_DIR / "audio" / "anuraset"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
MAX_PER_SPECIES = 200
STORAGE_CAP_GB = 10.0
START_T = time.time()


In [ ]:
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta', 1161364),
    ('Caiman yacare', '116570', 'Reptilia', 116570),
    ('Leptodactylus luctator', '1176823', 'Amphibia', 1176823),
    ('Adenomera guarani', '1491113', 'Amphibia', 1491113),
    ('Lysapsus limellum', '1595929', 'Amphibia', 1595929),
    ('Equus caballus', '209233', 'Mammalia', 209233),
    ('Leptodactylus syphax', '22930', 'Amphibia', 22930),
    ('Leptodactylus mystacinus', '22956', 'Amphibia', 22956),
    ('Leptodactylus podicipinus', '22961', 'Amphibia', 22961),
    ('Leptodactylus elenae', '22967', 'Amphibia', 22967),
    ('Leptodactylus fuscus', '22973', 'Amphibia', 22973),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia', 22983),
    ('Leptodactylus petersii', '22985', 'Amphibia', 22985),
    ('Physalaemus centralis', '23150', 'Amphibia', 23150),
    ('Physalaemus albifrons', '23154', 'Amphibia', 23154),
    ('Physalaemus albonotatus', '23158', 'Amphibia', 23158),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia', 23176),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia', 23724),
    ('Scinax nasicus', '24279', 'Amphibia', 24279),
    ('Scinax fuscovarius', '24285', 'Amphibia', 24285),
    ('Scinax fuscomarginatus', '24287', 'Amphibia', 24287),
    ('Scinax acuminatus', '24321', 'Amphibia', 24321),
    ('Quesada gigas', '244024', 'Insecta', 244024),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia', 25073),
    ('Elachistocleis bicolor', '25092', 'Amphibia', 25092),
    ('Dermatonotus muelleri', '25214', 'Amphibia', 25214),
    ('Physalaemus biligonigerus', '326272', 'Amphibia', 326272),
    ('Panthera onca', '41970', 'Mammalia', 41970),
    ('Alouatta caraya', '43435', 'Mammalia', 43435),
    ('Canis familiaris', '47144', 'Mammalia', 47144),
    ('Insect son01', '47158son01', 'Insecta', 47158),
    ('Insect son02', '47158son02', 'Insecta', 47158),
    ('Insect son03', '47158son03', 'Insecta', 47158),
    ('Insect son04', '47158son04', 'Insecta', 47158),
    ('Insect son05', '47158son05', 'Insecta', 47158),
    ('Insect son06', '47158son06', 'Insecta', 47158),
    ('Insect son07', '47158son07', 'Insecta', 47158),
    ('Insect son08', '47158son08', 'Insecta', 47158),
    ('Insect son09', '47158son09', 'Insecta', 47158),
    ('Insect son10', '47158son10', 'Insecta', 47158),
    ('Insect son11', '47158son11', 'Insecta', 47158),
    ('Insect son12', '47158son12', 'Insecta', 47158),
    ('Insect son13', '47158son13', 'Insecta', 47158),
    ('Insect son14', '47158son14', 'Insecta', 47158),
    ('Insect son15', '47158son15', 'Insecta', 47158),
    ('Insect son16', '47158son16', 'Insecta', 47158),
    ('Insect son17', '47158son17', 'Insecta', 47158),
    ('Insect son18', '47158son18', 'Insecta', 47158),
    ('Insect son19', '47158son19', 'Insecta', 47158),
    ('Insect son20', '47158son20', 'Insecta', 47158),
    ('Insect son21', '47158son21', 'Insecta', 47158),
    ('Insect son22', '47158son22', 'Insecta', 47158),
    ('Insect son23', '47158son23', 'Insecta', 47158),
    ('Insect son24', '47158son24', 'Insecta', 47158),
    ('Insect son25', '47158son25', 'Insecta', 47158),
    ('Physalaemus nattereri', '476521', 'Amphibia', 476521),
    ('Sapajus cay', '516975', 'Mammalia', 516975),
    ('Pithecopus azureus', '517063', 'Amphibia', 517063),
    ('Boana lundii', '555123', 'Amphibia', 555123),
    ('Boana punctata', '555145', 'Amphibia', 555145),
    ('Boana raniceps', '555146', 'Amphibia', 555146),
    ('Ameerega picta', '64898', 'Amphibia', 64898),
    ('Dendropsophus minutus', '65377', 'Amphibia', 65377),
    ('Dendropsophus nanus', '65380', 'Amphibia', 65380),
    ('Pseudis platensis', '66971', 'Amphibia', 66971),
    ('Rhinella diptycha', '67107', 'Amphibia', 67107),
    ('Trachycephalus typhonius', '67252', 'Amphibia', 67252),
    ('Leptodactylus macrosternum', '70711', 'Amphibia', 70711),
    ('Plecturocebus pallescens', '738183', 'Mammalia', 738183),
    ('Bos taurus', '74113', 'Mammalia', 74113),
    ('Mico melanurus', '74580', 'Mammalia', 74580),
    ('Prionacris erosa', '760266', 'Insecta', 760266),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves', 17431),
    ('Mustelirallus albicollis', 'astcra1', 'Aves', 508907),
    ('Crax fasciolata', 'bafcur1', 'Aves', 2046),
    ('Micrastur ruficollis', 'baffal1', 'Aves', 4699),
    ('Coereba flaveola', 'banana', 'Aves', 10199),
    ('Thamnophilus doliatus', 'barant1', 'Aves', 15764),
    ('Procnias nudicollis', 'batbel1', 'Aves', 8854),
    ('Ara ararauna', 'baymac', 'Aves', 19018),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves', 6893),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves', 558564),
    ('Donacobius atricapilla', 'bkcdon', 'Aves', 116877),
    ('Aratinga nenday', 'bkhpar', 'Aves', 367562),
    ('Busarellus nigricollis', 'blchaw1', 'Aves', 5346),
    ('Spizaetus tyrannus', 'blheag1', 'Aves', 5291),
    ('Tityra cayana', 'blttit1', 'Aves', 8830),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves', 16016),
    ('Megarynchus pitangua', 'bobfly1', 'Aves', 16737),
    ('Progne tapera', 'brcmar1', 'Aves', 11870),
    ('Tyto furcata', 'brnowl', 'Aves', 1578502),
    ('Momotus momota', 'bucmot4', 'Aves', 204447),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves', 367564),
    ('Amazona aestiva', 'bufpar', 'Aves', 18978),
    ('Theristicus caudatus', 'bunibi1', 'Aves', 3766),
    ('Athene cunicularia', 'burowl', 'Aves', 19975),
    ('Colaptes campestris', 'camfli1', 'Aves', 18262),
    ('Ortalis canicollis', 'chacha1', 'Aves', 2088),
    ('Mimus saturninus', 'chbmoc1', 'Aves', 14878),
    ('Gnorimopsar chopi', 'chobla1', 'Aves', 10723),
    ('Conirostrum speciosum', 'chvcon1', 'Aves', 10013),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves', 10921),
    ('Micrastur semitorquatus', 'coffal1', 'Aves', 4698),
    ('Nyctidromus albicollis', 'compau', 'Aves', 19627),
    ('Nyctibius griseus', 'compot1', 'Aves', 19667),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves', 12710),
    ('Pachyramphus validus', 'crebec1', 'Aves', 8681),
    ('Taoniscus nanus', 'dwatin1', 'Aves', 20714),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves', 72954),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves', 17231),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves', 144880),
    ('Glaucidium brasilianum', 'fepowl', 'Aves', 19822),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves', 14344),
    ('Myiothlypis flaveola', 'flawar1', 'Aves', 201223),
    ('Tyrannus savana', 'fotfly', 'Aves', 16793),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves', 17076),
    ('Hylocharis chrysura', 'gilhum1', 'Aves', 5979),
    ('Aramides ypecaha', 'giwrai1', 'Aves', 460),
    ('Chionomesa fimbriata', 'glteme1', 'Aves', 1289639),
    ('Saltator coerulescens', 'grasal3', 'Aves', 9850),
    ('Crotophaga major', 'greani1', 'Aves', 1970),
    ('Taraba major', 'greant1', 'Aves', 15957),
    ('Myiopagis viridicata', 'greela', 'Aves', 16892),
    ('Pitangus sulphuratus', 'grekis', 'Aves', 16956),
    ('Nyctibius grandis', 'grepot1', 'Aves', 19680),
    ('Phacellodomus ruber', 'gretho2', 'Aves', 11632),
    ('Tringa melanoleuca', 'greyel', 'Aves', 3892),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves', 3302),
    ('Eucometis penicillata', 'grhtan1', 'Aves', 10698),
    ('Aramides cajaneus', 'gycwor1', 'Aves', 513889),
    ('Anhima cornuta', 'horscr1', 'Aves', 6908),
    ('Passer domesticus', 'houspa', 'Aves', 13858),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves', 18938),
    ('Elaenia spectabilis', 'larela1', 'Aves', 16734),
    ('Elaenia chiriquensis', 'lesela1', 'Aves', 578460),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves', 10555),
    ('Aramus guarauna', 'limpki', 'Aves', 7),
    ('Dryocopus lineatus', 'linwoo1', 'Aves', 17858),
    ('Coccycua minuta', 'litcuc2', 'Aves', 72740),
    ('Setopagis parvula', 'litnig1', 'Aves', 367507),
    ('Pyrrhura frontalis', 'mabpar', 'Aves', 19162),
    ('Cercomacra melanaria', 'magant1', 'Aves', 15737),
    ('Cissopis leverianus', 'magtan2', 'Aves', 72727),
    ('Polioptila dumicola', 'masgna1', 'Aves', 7509),
    ('Chordeiles nacunda', 'nacnig1', 'Aves', 19661),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves', 1506288),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves', 11511),
    ('Icterus croconotus', 'orbtro3', 'Aves', 62564),
    ('Amazona amazonica', 'orwpar', 'Aves', 18982),
    ('Pandion haliaetus', 'osprey', 'Aves', 116999),
    ('Synallaxis albescens', 'pabspi1', 'Aves', 10999),
    ('Furnarius leucopus', 'palhor3', 'Aves', 11281),
    ('Thraupis palmarum', 'paltan1', 'Aves', 10297),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves', 1982),
    ('Patagioenas picazuro', 'picpig2', 'Aves', 3102),
    ('Legatus leucophaius', 'pirfly1', 'Aves', 17312),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves', 73493),
    ('Inezia inornata', 'platyr1', 'Aves', 16344),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves', 8484),
    ('Theristicus caerulescens', 'pluibi1', 'Aves', 3768),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves', 8483),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves', 16273),
    ('Ara chloropterus', 'ragmac1', 'Aves', 19016),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves', 11201),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves', 10310),
    ('Gallus gallus', 'redjun', 'Aves', 882),
    ('Cariama cristata', 'relser1', 'Aves', 14),
    ('Megaceryle torquata', 'rinkin1', 'Aves', 2552),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves', 145267),
    ('Rupornis magnirostris', 'roahaw', 'Aves', 201041),
    ('Turdus rufiventris', 'rubthr1', 'Aves', 12738),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves', 11718),
    ('Casiornis rufus', 'rufcas2', 'Aves', 17102),
    ('Conopophaga lineata', 'rufgna3', 'Aves', 578313),
    ('Furnarius rufus', 'rufhor2', 'Aves', 11275),
    ('Antrostomus rufus', 'rufnig1', 'Aves', 201066),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves', 11624),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves', 17026),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves', 16833),
    ('Tigrisoma lineatum', 'ruther1', 'Aves', 5048),
    ('Galbula ruficauda', 'rutjac1', 'Aves', 1468),
    ('Arremon flavirostris', 'sabspa1', 'Aves', 10064),
    ('Sicalis flaveola', 'saffin', 'Aves', 9864),
    ('Thraupis sayaca', 'saytan1', 'Aves', 10293),
    ('Columbina squammata', 'scadov1', 'Aves', 3564),
    ('Pionus maximiliani', 'schpar1', 'Aves', 19094),
    ('Phaethornis eurynome', 'scther1', 'Aves', 5622),
    ('Myiarchus ferox', 'shcfly1', 'Aves', 16006),
    ('Accipiter striatus', 'shshaw', 'Aves', 5097),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves', 19645),
    ('Ramphocelus carbo', 'sibtan2', 'Aves', 10056),
    ('Crotophaga ani', 'smbani', 'Aves', 1971),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves', 20570),
    ('Cacicus solitarius', 'sobcac1', 'Aves', 10365),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves', 16972),
    ('Myiozetetes similis', 'socfly1', 'Aves', 16842),
    ('Synallaxis frontalis', 'sofspi1', 'Aves', 10996),
    ('Corythopis delalandi', 'souant1', 'Aves', 17264),
    ('Vanellus chilensis', 'soulap1', 'Aves', 4867),
    ('Chauna torquata', 'souscr1', 'Aves', 6910),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves', 15959),
    ('Synallaxis spixi', 'spispi1', 'Aves', 10915),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves', 1584760),
    ('Piaya cayana', 'squcuc1', 'Aves', 1758),
    ('Dendroplex picus', 'stbwoo2', 'Aves', 72806),
    ('Tapera naevia', 'strcuc1', 'Aves', 1989),
    ('Butorides striata', 'strher2', 'Aves', 62528),
    ('Asio clamator', 'strowl1', 'Aves', 558468),
    ('Eupetomena macroura', 'swthum1', 'Aves', 6065),
    ('Chiroxiphia caudata', 'swtman1', 'Aves', 14306),
    ('Crypturellus tataupa', 'tattin1', 'Aves', 20587),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves', 7480),
    ('Ramphastos toco', 'toctou1', 'Aves', 18793),
    ('Tyrannus melancholicus', 'trokin', 'Aves', 16787),
    ('Megascops choliba', 'trsowl', 'Aves', 19788),
    ('Crypturellus undulatus', 'undtin1', 'Aves', 20592),
    ('Thamnophilus caerulescens', 'varant1', 'Aves', 15757),
    ('Jacana jacana', 'watjac1', 'Aves', 4580),
    ('Pyriglena maura', 'wesfie1', 'Aves', 1286886),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves', 6898),
    ('Biatas nigropectus', 'whbant2', 'Aves', 15902),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves', 201224),
    ('Melanerpes candidus', 'whiwoo1', 'Aves', 18183),
    ('Synallaxis albilora', 'whlspi1', 'Aves', 10992),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves', 8469),
    ('Leptotila verreauxi', 'whtdov', 'Aves', 3280),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves', 17786),
    ('Caracara plancus', 'y00678', 'Aves', 4715),
    ('Paroaria capitata', 'yebcar', 'Aves', 10257),
    ('Elaenia flavogaster', 'yebela1', 'Aves', 16714),
    ('Primolius auricollis', 'yecmac', 'Aves', 73272),
    ('Brotogeris chiriri', 'yecpar', 'Aves', 19215),
    ('Daptrius chimachima', 'yehcar1', 'Aves', 1432779),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves', 16567),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name", "inat_taxon_id"])
SCI_LC_TO_LABEL = {str(s).lower(): l for s, l in zip(species_df['scientific_name'], species_df['primary_label'])}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))
SCI_LC_SET = set(SCI_LC_TO_LABEL.keys())
# Amphibia subset
amphibia_df = species_df[species_df['class_name'] == 'Amphibia']
print(f"BC2026: {len(species_df)} species, Amphibia: {len(amphibia_df)}")


In [ ]:
# AnuraSet annotation from Zenodo (official, doi: 10.5281/zenodo.8342596)
import urllib.request, zipfile

ZENODO_BASE = "https://zenodo.org/records/8342596/files"

# 1. Download species.csv (BC2026 match に使う scientific_name list)
species_csv_path = OUT_DIR / "anuraset_species.csv"
print(f"DL species.csv...")
urllib.request.urlretrieve(f"{ZENODO_BASE}/species.csv?download=1", species_csv_path)
species_meta = pd.read_csv(species_csv_path)
print(f"  ✓ {len(species_meta)} species")
print(f"  Columns: {species_meta.columns.tolist()}")
print(species_meta.head(5).to_string())

# 2. Download weak_labels.csv (1-min level annotations, fine for our purpose)
weak_path = OUT_DIR / "anuraset_weak_labels.csv"
print(f"\nDL weak_labels.csv...")
urllib.request.urlretrieve(f"{ZENODO_BASE}/weak_labels.csv?download=1", weak_path)
weak_df = pd.read_csv(weak_path)
print(f"  ✓ {len(weak_df)} annotations")
print(f"  Columns: {weak_df.columns.tolist()}")
print(weak_df.head(3).to_string())

# 3. Download strong_labels.zip (precise segment annotations)
strong_zip = OUT_DIR / "strong_labels.zip"
strong_dir = OUT_DIR / "strong_labels"
print(f"\nDL strong_labels.zip...")
try:
    urllib.request.urlretrieve(f"{ZENODO_BASE}/strong_labels.zip?download=1", strong_zip)
    strong_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(strong_zip) as zf:
        zf.extractall(strong_dir)
    print(f"  Extracted to {strong_dir}")
    # List
    for f in strong_dir.rglob("*"):
        if f.is_file():
            print(f"    {f.relative_to(strong_dir)}  ({f.stat().st_size/1024:.1f} KB)")
except Exception as e:
    print(f"  strong_labels DL err: {str(e)[:100]}")
    strong_dir = None

# Use weak_labels as primary annotation (full coverage, faster)
annotation_df = weak_df

# Map species in annotation to scientific names
if "scientific_name" not in annotation_df.columns and "species" in species_meta.columns:
    # weak_labels may have species_code, join via species_meta
    species_code_col = next((c for c in annotation_df.columns if c.lower() in ["species_code", "code", "label"]), None)
    if species_code_col:
        sci_lookup = dict(zip(species_meta[species_meta.columns[0]], species_meta["species"]))
        annotation_df["scientific_name"] = annotation_df[species_code_col].map(sci_lookup)
print(f"\nFinal annotation: {len(annotation_df)} rows, cols: {annotation_df.columns.tolist()}")


In [ ]:
def safe_dir(name):
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(name))

def check_storage_gb():
    try:
        return sum(f.stat().st_size for f in AUDIO_DIR.rglob("*") if f.is_file()) / 1e9
    except: return 0.0

ANURA_CANDIDATES = [
    Path("/kaggle/input/datasets/bengtlueers/anuraset-v2-raw"),
    Path("/kaggle/input/anuraset-v2-raw"),
]
anura_root = next((p for p in ANURA_CANDIDATES if p.exists()), None)
if anura_root is None:
    raise RuntimeError(f"AnuraSet NOT MOUNTED: {ANURA_CANDIDATES}")

data_dir = None
for cand in [anura_root / "AnuraSet_v2.0.0_raw", anura_root]:
    if cand.exists() and any(c.is_dir() for c in cand.iterdir()):
        data_dir = cand; break
print(f"AnuraSet data_dir: {data_dir}")

# === AnuraSet uses WIDE-format multi-label annotation ===
# species.csv: FAMILY, SPECIES (scientific_name), CODE (6-letter)
# weak_labels.csv: MONITORING_SITE, AUDIO_FILE_ID, SPECIES_XXX (one col per species), 0-3 calling activity

# Build CODE → scientific_name map from species.csv
code_to_sci = {}
code_col = next((c for c in species_meta.columns if c.upper() == "CODE"), None)
sci_col_meta = next((c for c in species_meta.columns if c.upper() == "SPECIES"), None)
print(f"species.csv columns: {species_meta.columns.tolist()}")
print(f"  CODE col: {code_col}, SPECIES col: {sci_col_meta}")
if not code_col or not sci_col_meta:
    raise RuntimeError(f"species.csv missing CODE or SPECIES columns: {species_meta.columns.tolist()}")
for _, r in species_meta.iterrows():
    code_to_sci[str(r[code_col]).upper()] = str(r[sci_col_meta])
print(f"  Built CODE→sci map: {len(code_to_sci)} species")

# weak_labels.csv: identify species columns (SPECIES_*)
species_cols = [c for c in weak_df.columns if c.startswith("SPECIES_")]
print(f"\nweak_labels.csv: {len(weak_df)} rows, {len(species_cols)} species columns")
print(f"  Sample species cols: {species_cols[:5]}")

# For each species column, get scientific_name + check BC2026 match
bc26_species_cols = []  # list of (species_col, primary_label, sci_lc)
for col in species_cols:
    code = col.replace("SPECIES_", "").upper()
    sci = code_to_sci.get(code, "")
    sci_lc = sci.lower()
    if sci_lc in SCI_LC_SET:
        primary_label = SCI_LC_TO_LABEL[sci_lc]
        bc26_species_cols.append((col, primary_label, sci_lc, code))

print(f"\nBC2026 match species: {len(bc26_species_cols)}")
for col, lbl, sci, code in bc26_species_cols[:20]:
    n_pos = (weak_df[col] > 0).sum()
    print(f"  {code:8s} → {sci:35s} → {lbl}  ({n_pos} files positive)")
if not bc26_species_cols:
    print(f"\n  All AnuraSet species: {[code_to_sci.get(c.replace('SPECIES_',''), '?') for c in species_cols[:10]]}")
    print(f"  BC2026 Amphibia sci_name sample: {[s for s in SCI_LC_SET if s.startswith('boa') or s.startswith('lep')][:10]}")
    raise RuntimeError("0 BC2026 species matched in AnuraSet")

# For each BC2026 species, find audio files where this species has positive activity
all_metadata = []
n_copied = 0
audio_id_col = "AUDIO_FILE_ID"
site_col = "MONITORING_SITE"

for col, primary_label, sci_lc, code in tqdm(bc26_species_cols, desc="AnuraSet species"):
    if check_storage_gb() > STORAGE_CAP_GB: break
    # Get audio files with this species positive
    pos_rows = weak_df[weak_df[col] > 0].head(MAX_PER_SPECIES)
    if len(pos_rows) == 0: continue

    dst_dir = AUDIO_DIR / safe_dir(primary_label)
    dst_dir.mkdir(parents=True, exist_ok=True)

    for _, r in pos_rows.iterrows():
        if check_storage_gb() > STORAGE_CAP_GB: break
        audio_id = str(r[audio_id_col])
        site = str(r[site_col])
        # bengtlueers structure: <site_no_zero>/{audio_id}.wav
        # site in weak_labels = "INCT04", file in bengtlueers = "INCT4"
        site_normalized = site.replace("INCT0", "INCT")  # remove leading zero
        # Try multiple path patterns
        candidate_paths = [
            data_dir / site / f"{audio_id}.wav",
            data_dir / site_normalized / f"{audio_id}.wav",
            data_dir / f"{audio_id}.wav",
        ]
        # Fallback: rglob
        src = next((p for p in candidate_paths if p.exists()), None)
        if src is None:
            for p in data_dir.rglob(f"{audio_id}.wav"):
                src = p; break
        if src is None: continue

        dst = dst_dir / src.name
        if dst.exists() and dst.stat().st_size > 100: continue
        try:
            shutil.copy2(src, dst)
            n_copied += 1
            all_metadata.append({
                "filename": str(dst.relative_to(OUT_DIR)),
                "primary_label": primary_label,
                "scientific_name": sci_lc,
                "source": "anuraset",
                "class_name": "Amphibia",
                "activity": int(r[col]),
                "audio_id": audio_id,
                "site": site,
                "file_size_mb": dst.stat().st_size / 1e6,
            })
        except Exception as e:
            pass

print(f"\nCopied: {n_copied} files from {len(bc26_species_cols)} species")
print(f"Storage: {check_storage_gb():.2f} GB, time: {(time.time()-START_T)/60:.1f} min")

if all_metadata:
    pd.DataFrame(all_metadata).to_csv(OUT_DIR / "metadata.csv", index=False)


In [ ]:
import json
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

# Clean up temporary annotation files before upload (avoid Kaggle SDK token bug
# from too many file uploads)
print("Cleaning OUT_DIR for upload...")
for f in OUT_DIR.iterdir():
    # Keep only audio/ dir and metadata.csv
    if f.name in ("audio", "metadata.csv"):
        continue
    try:
        if f.is_file():
            f.unlink()
            print(f"  removed file: {f.name}")
        elif f.is_dir():
            import shutil
            shutil.rmtree(f)
            print(f"  removed dir: {f.name}")
    except Exception as e:
        print(f"  rm err {f.name}: {e}")

print(f"\nOUT_DIR after cleanup:")
for f in OUT_DIR.iterdir():
    print(f"  {f.name}")

USER = "maekeso"; SLUG = "birdclef2026-exp047-anuraset-extracted"; TITLE = "BirdCLEF2026 exp047 AnuraSet Extracted"
DRY_RUN = False
if not DRY_RUN:
    meta = {"title": TITLE, "id": f"{USER}/{SLUG}", "licenses": [{"name": "other"}]}
    (OUT_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
    try:
        api.dataset_create_new(folder=str(OUT_DIR), public=False, dir_mode="zip", quiet=False)
        print("OK new dataset created")
    except Exception as e:
        print(f"create_new err: {str(e)[:200]}")
        try:
            api.dataset_create_version(folder=str(OUT_DIR), version_notes="AnuraSet extracted v2",
                                        dir_mode="zip", quiet=False)
            print("OK version created")
        except Exception as e2:
            print(f"create_version err: {str(e2)[:200]}")
    print(f"\nURL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
